# PyMIF notebook 10 - fuse a mosaic (3D and 2D MIP)

Some acquisitions are saved as many separate **tiles**, stacks acquired at different stage positions, instead of one image. `pymif.mosaic_manager` reads them lazily and **fuses** them into one OME-Zarr image.

This notebook:

1. creates a small synthetic *raw Luxendo acquisition* (two illumination sides, 2 x 2 tiles each, two dyes), so it runs without real data;
2. loads it with `LuxendoMosaicManager` and checks the output size before fusing;
3. fuses a **3D volume** and a **2D maximum projection (MIP)**;
4. makes a `sum` overview that shows where the tiles overlap;
5. passes the same OME-Zarr options as `pymif 2zarr` (chunks, shards, zarr format, channel names...);
6. shows the equivalent command lines and a template for real data.

In [ ]:
from pathlib import Path
import json
import logging
import shutil

import h5py
import matplotlib.pyplot as plt
import numpy as np

from pymif.mosaic_manager import LuxendoMosaicManager, plan_fusion

logging.basicConfig(format="%(message)s")
logging.getLogger("pymif").setLevel(logging.INFO)

OUT = Path("pymif_tutorial_output") / "10_mosaic"
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)

## 1. A synthetic raw Luxendo acquisition

A raw Luxendo acquisition has one folder per tile and dye in `raw/`, each with a `Cam_*.lux.h5` file (the stack, dataset `Data`) and a `.json` file. The JSON holds the voxel size, the image size and the stage position (`affine_to_sample`).

The cell below cuts a synthetic sample (bright blobs on a dim background) into overlapping tiles and writes them in that layout. The left illumination side (`-L`) covers the left part of the sample and the right side (`-R`) the right part, with an overlap in the middle. The right side is made a bit dimmer, as happens with real light sheets far from their side.

In [ ]:
VOXEL = (3.0, 2.0, 2.0)          # z, y, x in micrometer
TILE = (24, 64, 64)              # z, y, x voxels per tile
rng = np.random.default_rng(0)

# The "true" sample: 24 x 112 x 208 voxels
sample = np.full((24, 112, 208), 100.0)
zz, yy, xx = np.mgrid[:24, :112, :208]
for _ in range(60):
    z, y, x = rng.uniform((0, 0, 0), sample.shape)
    sample += 3000 * np.exp(-((zz - z) ** 2 / 8 + (yy - y) ** 2 / 20 + (xx - x) ** 2 / 20))


def write_tile(folder, data, dye, side, stack, y0, x0):
    """Write one tile like the Luxendo software: .lux.h5 + .json with the stage position."""
    folder.mkdir(parents=True)
    with h5py.File(folder / "Cam_bottom_00000.lux.h5", "w") as f:
        f.create_dataset("Data", data=data.astype(np.uint16), chunks=(8, 32, 32))
    nz, ny, nx = data.shape
    info = {
        "channel_description": f"{dye}-{side}",
        "stack": stack,
        "stack_description": f"{side} side",
        "voxel_size_um": {"width": VOXEL[2], "height": VOXEL[1], "depth": VOXEL[0]},
        "image_size_vx": {"width": nx, "height": ny, "depth": nz},
        # the stage position is the tile centre; z planes step by -3 um
        "affine_to_sample": [
            {"matrix": np.eye(3).tolist(), "translation": [-(nx - 1) / 2, -(ny - 1) / 2, 0]},
            {"matrix": [[VOXEL[2], 0, 0], [0, VOXEL[1], 0], [0, 0, -VOXEL[0]]],
             "translation": [x0 * VOXEL[2] + (nx - 1) / 2 * VOXEL[2], y0 * VOXEL[1] + (ny - 1) / 2 * VOXEL[1], 0.0]},
        ],
    }
    (folder / "Cam_bottom_00000.json").write_text(json.dumps({"processingInformation": info}))


acq = OUT / "2026-10-14_120000"
for side, stack_id, x_starts, gain in (("L", 1, (0, 56), 1.0), ("R", 2, (88, 144), 0.6)):
    for ix, x0 in enumerate(x_starts):
        for iy, y0 in enumerate((0, 48)):
            for dye, dye_gain in (("nuclei-488", 1.0), ("cFos-647", 0.4)):
                stack = f"{stack_id}-x{ix:02d}-y{iy:02d}"
                tile = sample[:, y0 : y0 + TILE[1], x0 : x0 + TILE[2]] * gain * dye_gain
                write_tile(acq / "raw" / f"stack_{stack}_channel_{dye}-{side}_obj_bottom",
                           tile, dye, side, stack, y0, x0)

print(len(list((acq / "raw").iterdir())), "tile folders written to", acq / "raw")

## 2. Load the tiles

`LuxendoMosaicManager` reads only the JSON files (and one `.h5` header), so loading is fast even over a network drive. Both illumination sides of a dye become **one channel**; the side is kept as the tile `group`.

In [ ]:
mosaic = LuxendoMosaicManager(acq)

print("tiles:   ", len(mosaic.tiles))
print("channels:", mosaic.channels)
print("groups:  ", mosaic.groups)
print("extent (z, y, x) um:", mosaic.bounding_box())

t = mosaic.tiles[0]
print("\nfirst tile:", t.channel, t.group, t.shape, "voxel", t.voxel_size, "origin", t.origin, t.attrs["stack"])

`plan_fusion` takes the fusion arguments and returns the output shape and size **without reading any pixel data**. Use it to choose the downsampling (`ds`) before a long fusion.

In [ ]:
for ds in [(1, 1, 1), (2, 4, 4)]:
    print("3D ", ds, plan_fusion(mosaic.tiles, ds=ds).summary())
print("MIP", (1, 1, 1), plan_fusion(mosaic.tiles, projection="max").summary())

## 3. 3D fusion

`fuse()` writes an OME-Zarr and returns it as a `ZarrManager`.

- `ds=(z, y, x)` downsamples while reading the tiles: each output voxel is the mean of its block.
- `blend="midline"` uses the brighter value within each side (`max`) and, where the two sides overlap, the left side left of the middle of the overlap and the right side right of it. Each side is then shown where its light sheet is sharpest.
- `levels` and `downscale_factor` build the pyramid; `workers` reads tiles in parallel processes.

In [ ]:
fused_3d = mosaic.fuse(
    OUT / "mosaic_3D.ome.zarr",
    ds=(1, 2, 2),
    blend="midline",
    levels=2,
    downscale_factor=(1, 2, 2),
    workers=2,
    overwrite=True,
)
print(fused_3d.metadata["axes"], [lvl.shape for lvl in fused_3d.data], fused_3d.metadata["scales"])

volume = fused_3d.data[0].compute()                     # (c, z, y, x)
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, c in zip(axs, range(volume.shape[0])):
    ax.imshow(volume[c, volume.shape[1] // 2], cmap="gray")
    ax.set_title(f"{fused_3d.metadata['channel_names'][c]}, middle plane")
plt.show()

## 4. 2D maximum projection (MIP)

`projection="max"` projects every tile along z at full resolution and then downsamples in y/x, so the output is a small 2D image even for a large acquisition. `projection="sum"` makes a sum projection instead.

In [ ]:
mip = mosaic.fuse(
    OUT / "mosaic_MIP.ome.zarr",
    projection="max",
    ds=(1, 1, 1),
    blend="midline",
    overwrite=True,
)
img = mip.data[0].compute()                              # (c, y, x)
print(mip.metadata["axes"], img.shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, c in zip(axs, range(img.shape[0])):
    ax.imshow(img[c], cmap="magma")
    ax.set_title(f"MIP {mip.metadata['channel_names'][c]}")
plt.show()

## 5. Where do the tiles overlap?

`blend="sum"` adds the overlapping tiles, so the overlaps show up as brighter bands (output `float32`).

In [ ]:
summed = mosaic.fuse(OUT / "tiles_sum.ome.zarr", projection="max", blend="sum",
                     channels=["nuclei-488"], overwrite=True)

plt.figure(figsize=(6, 3))
plt.imshow(summed.data[0].compute(), cmap="gray")
plt.title("sum: overlaps are brighter")
plt.show()

## 6. The same output options as `pymif 2zarr`

The fused result is written with PyMIF's standard writer, so every `to_zarr` option works exactly as for any other conversion: `chunks`, `shards`, `shard_target_mb`, `shard_exclude_axes`, `zarr_format`, `compressor`, `drop_singleton`, `num_workers`, ... In addition:

- `channels` picks the tile channels to fuse, and `channel_names` / `channel_colors` name and colour the output channels;
- `crop` writes only part of the output, in output voxels (after `ds`). For a projection, a `z` crop limits the planes that are projected.

In [ ]:
custom = mosaic.fuse(
    OUT / "mosaic_custom.ome.zarr",
    ds=(1, 2, 2),
    blend="midline",
    channels=["cFos-647", "nuclei-488"],          # output order
    channel_names=["cfos", "nuclei"],
    channel_colors=["magenta", "00FF00"],
    crop={"y": slice(10, 50), "x": slice(20, 90)},
    levels=2,
    chunks=(1, 8, 32, 32),                        # c, z, y, x
    shards="auto",
    zarr_format=3,
    num_workers=2,
    overwrite=True,
)
print(custom.metadata["channel_names"], custom.metadata["channel_colors"])
print([lvl.shape for lvl in custom.data])

# one channel only: the channel axis is dropped, as with pymif 2zarr (drop_singleton=False keeps it)
single = mosaic.fuse(OUT / "nuclei_only.ome.zarr", projection="max", channels=["nuclei-488"], overwrite=True)
print(single.metadata["axes"])

## 7. Command line

The same fusions from a terminal (`pymif mosaic2zarr -h` lists every option). All `2zarr` options are available with the same flags (`-cs`, `-ms`, `-cn`, `-cc`, `-zf`, `-df`, `-nl`, `-s`, `-sh`, `-stm`, `-sea`, `-ds`, `-nw`); `-s/--subset` crops the fused output and selects channels.

```console
# 3D, downsampled 4 (z) x 8 (y) x 8 (x), midline blend, 3 pyramid levels
pymif mosaic2zarr -i /path/to/2026-10-14_103957 -z /path/to/acq_3D.ome.zarr -rds 4 8 8 -b midline -nl 3

# 2D maximum projection, downsampled 8 x 8 in y/x
pymif mosaic2zarr -i /path/to/2026-10-14_103957 -z /path/to/acq_MIP.ome.zarr -rds 1 8 8 -p max -b midline

# where are the tiles? (overlaps are brighter)
pymif mosaic2zarr -i /path/to/2026-10-14_103957 -z /path/to/acq_tiles.ome.zarr -rds 1 8 8 -p max -b sum

# many acquisitions from a CSV file (columns: see pymif batch-mosaic2zarr -h)
pymif batch-mosaic2zarr -i mosaics.csv
```

In napari, `Plugins > PyMIF > Mosaic Fusion` does the same interactively, and `mosaic.visualize()` shows every tile as its own layer at its stage position.

## 8. Template for real data

Raw tiles have no pyramid, so every fusion reads the full-resolution data whatever `ds` is. On a network drive, the read speed sets the run time: check the output size with `plan_fusion` first, and consider `temp_dir=` on a local disk for the temporary fused level 0.

In [ ]:
# acquisition = r"Y:/path/to/2026-10-14_103957"       # the folder that contains raw/
# mosaic = LuxendoMosaicManager(acquisition)
# print(plan_fusion(mosaic.tiles, ds=(4, 8, 8), blend="midline").summary())
#
# mosaic.fuse(r"D:/fused/acq_3D.ome.zarr", ds=(4, 8, 8), blend="midline", levels=3, workers=4)
# mosaic.fuse(r"D:/fused/acq_MIP.ome.zarr", ds=(1, 8, 8), projection="max", blend="midline", workers=4)